## Average Accuracy Calculation


In [1]:
import os
import json
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score
import numpy as np
import pandas as pd

def read_json(path):
    with open(path, 'r', encoding="utf-8") as f:
        data = json.load(f)
    return data

def write_json(data, path):
    if not os.path.exists(os.path.dirname(path)):
        os.makedirs(os.path.dirname(path))
    with open(path, 'w', encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
        

In [16]:
#fewrel_t5 results-si
experiments = {'run_1':'/Users/sefika/phd_projects/llm-catastrophic-re/fewrel_corrected_results/tmlr/mas/canonical/model_1_forgetting_matrix.json',
               'run_2': '/Users/sefika/phd_projects/llm-catastrophic-re/fewrel_corrected_results/tmlr/mas/canonical/model_2_forgetting_matrix.json' ,
               'run_3': '/Users/sefika/phd_projects/llm-catastrophic-re/fewrel_corrected_results/tmlr/mas/canonical/model_3_forgetting_matrix.json',
               'run_4': '/Users/sefika/phd_projects/llm-catastrophic-re/fewrel_corrected_results/tmlr/mas/canonical/model_4_forgetting_matrix.json',
               'run_5': '/Users/sefika/phd_projects/llm-catastrophic-re/fewrel_corrected_results/tmlr/mas/canonical/model_5_forgetting_matrix.json'}

In [17]:
data1 = read_json(experiments['run_1'])
data2 = read_json(experiments['run_2'])
data3 = read_json(experiments['run_3'])
data4 = read_json(experiments['run_4'])
data5 = read_json(experiments['run_5'])

In [18]:
data1 = pd.DataFrame(data1)
data2 = pd.DataFrame(data2)
data3 = pd.DataFrame(data3)
data4 = pd.DataFrame(data4)
data5 = pd.DataFrame(data5)

In [19]:
combined = pd.concat([data1, data2, data3, data4, data5])


In [20]:
results = []
for base in range(1,9):
    total_task_accuracy =[]
    std = 0
    for task in range(1,base+1):
        acc = combined[(combined['base_task'] == base) & (combined['task'] == task)]['accuracy'].values

        if acc.size > 0:
            total_task_accuracy.append(sum(acc) / len(acc))
        if base == 1 and task == 1:
            std = np.std(acc) if len(acc) > 0 else 0
    if base == 1:
        row = {
            "base": base,
            "mean_accuracy": (sum(total_task_accuracy) / len(total_task_accuracy))*100 if total_task_accuracy else 0,
            "standard_deviation": std*100 if total_task_accuracy else 0
        }
        results.append(row)
    else:
        row = {
            "base": base,
            "mean_accuracy": (sum(total_task_accuracy) / len(total_task_accuracy))*100 if total_task_accuracy else 0,
            "standard_deviation": (np.std(total_task_accuracy))*100 if total_task_accuracy else 0
        }
    results.append(row)

In [21]:
results_df = pd.DataFrame(results)
results_df.T



,0,1,2,3,4,5,6,7,8
base,1.000000,1.000000,2.000000,3.000000,4.000000,5.00000,6.000000,7.000000,8.000000
mean_accuracy,98.057143,98.057143,89.592857,86.200000,86.046429,85.28000,85.240476,85.116327,85.203571
standard_deviation,0.765853,0.765853,5.321429,6.306664,5.793490,5.82611,5.399807,5.749029,5.292914


In [9]:
results_df = pd.DataFrame(results)
results_df.T



,0,1,2,3,4,5,6,7,8
base,1.000000,1.000000,2.000000,3.000000,4.000000,5.000000,6.000000,7.000000,8.000000
mean_accuracy,98.057143,98.057143,89.585714,86.185714,86.039286,85.045714,85.030952,84.983673,85.123214
standard_deviation,0.765853,0.765853,5.314286,6.309403,5.801006,5.860789,5.499588,5.836546,5.359418


In [ ]:
results_df.to_csv('/Users/sefika/phd_projects/llm-catastrophic-re/fewrel_corrected_results/tmlr/mas/icl_tacred_acc.csv', index=False)